In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 4 Commit 1: Spark Session & Resource Configuration
# feat(task4): SparkSession with documented resource configuration
# Coursework Requirement:
# Resource configuration screenshot + Spark Session initialisation
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task4")
    .master("local[*]")

    .config("spark.driver.memory","10g")
    .config("spark.executor.memory","4g")
    .config("spark.driver.maxResultSize","2g")

    .config("spark.sql.shuffle.partitions","200")
    .config("spark.default.parallelism","200")

    .config("spark.memory.fraction","0.8")
    .config("spark.memory.storageFraction","0.3")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("="*80)
print("RESOURCE CONFIGURATION")
print("="*80)

configs = [

    "spark.driver.memory",

    "spark.executor.memory",

    "spark.driver.maxResultSize",

    "spark.sql.shuffle.partitions",

    "spark.default.parallelism",

    "spark.memory.fraction",

    "spark.memory.storageFraction"

]

for c in configs:

    print(f"{c:<45} : {spark.conf.get(c)}")

print()

print("Spark UI")

print(spark.sparkContext.uiWebUrl)

# ──────────────────────────────────────────────────────────────────────────────
# Project Directories
# ──────────────────────────────────────────────────────────────────────────────

BASE = "/content/drive/MyDrive/7006SCN"

PROCESSED = f"{BASE}/data/processed"

MODELS = f"{BASE}/models"

OUTPUT = f"{BASE}/outputs"

# ──────────────────────────────────────────────────────────────────────────────
# Load HMDA Dataset
# ──────────────────────────────────────────────────────────────────────────────

df_clean = spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print()

print("="*80)
print("HMDA DATASET LOADED")
print("="*80)

print(f"Rows    : {df_clean.count():,}")

print(f"Columns : {len(df_clean.columns)}")

# PIPELINE Loading
from pyspark.ml import PipelineModel

pipeline_model = PipelineModel.load(
    f"{MODELS}/preprocessing_pipeline"
)

print("Pipeline Loaded ✓")

# Generate Features
df_model = pipeline_model.transform(df_clean)

print("Feature Vector Created ✓")

# Loading Saved Models
from pyspark.ml.classification import (
    LogisticRegressionModel,
    DecisionTreeClassificationModel,
    RandomForestClassificationModel,
    GBTClassificationModel
)

lr_model = LogisticRegressionModel.load(
    f"{MODELS}/lr_cv_best"
)

dt_model = DecisionTreeClassificationModel.load(
    f"{MODELS}/dt_best"
)

rf_model = RandomForestClassificationModel.load(
    f"{MODELS}/rf_best"
)

gbt_model = GBTClassificationModel.load(
    f"{MODELS}/gbt_best"
)

print()

print("Saved Models Loaded")

print("✓ Logistic Regression")

print("✓ Decision Tree")

print("✓ Random Forest")

print("✓ Gradient Boosted Trees")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RESOURCE CONFIGURATION
spark.driver.memory                           : 10g
spark.executor.memory                         : 4g
spark.driver.maxResultSize                    : 2g
spark.sql.shuffle.partitions                  : 200
spark.default.parallelism                     : 200
spark.memory.fraction                         : 0.8
spark.memory.storageFraction                  : 0.3

Spark UI
http://9775fb37b770:4040


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 4 Commit 2: Caching, Persisting and Performance Benchmark
# feat(task4): MEMORY_AND_DISK caching with speed-up benchmark
#
# Coursework Requirement:
# Demonstrate repartitioning, caching and performance optimisation
# ──────────────────────────────────────────────────────────────────────────────

from pyspark import StorageLevel
from pyspark.sql.functions import monotonically_increasing_id

print("="*80)
print("COMMIT 2 : REPARTITIONING & CACHING")
print("="*80)

# -----------------------------------------------------------------------------
# Original Partitions
# -----------------------------------------------------------------------------

print(f"Original Partitions : {df_model.rdd.getNumPartitions()}")

# -----------------------------------------------------------------------------
# Repartition Dataset
# -----------------------------------------------------------------------------

df_repartitioned = (
    df_model
    .withColumn(
        "_bucket",
        (monotonically_increasing_id() % 200).cast("int")
    )
    .repartition(200, "_bucket")
    .drop("_bucket")
)

print(f"Repartitioned Dataset : {df_repartitioned.rdd.getNumPartitions()}")

# -----------------------------------------------------------------------------
# Cache Dataset
# -----------------------------------------------------------------------------

start = time.time()

df_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)

df_repartitioned.count()

cache_time = time.time() - start

print()
print(f"Cache Materialised : {cache_time:.2f} seconds")
print("Storage Level      : MEMORY_AND_DISK")

# -----------------------------------------------------------------------------
# Cold Read Benchmark
# -----------------------------------------------------------------------------

start = time.time()

cold_df = pipeline_model.transform(df_clean)

cold_df.count()

cold_time = time.time() - start

# -----------------------------------------------------------------------------
# Warm Read Benchmark
# -----------------------------------------------------------------------------

start = time.time()

df_repartitioned.count()

warm_time = time.time() - start

print()
print("="*80)
print("PERFORMANCE BENCHMARK")
print("="*80)

print(f"Cold Execution : {cold_time:.2f} sec")
print(f"Warm Execution : {warm_time:.2f} sec")

if warm_time > 0:
    print(f"Speed-up        : {cold_time / warm_time:.2f}x")

print("="*80)
print("COMMIT 2 COMPLETED")
print("="*80)

print("\nSpark UI")
print(spark.sparkContext.uiWebUrl)

COMMIT 2 : REPARTITIONING & CACHING
Original Partitions : 200
Repartitioned Dataset : 200

Cache Materialised : 3029.22 seconds
Storage Level      : MEMORY_AND_DISK

PERFORMANCE BENCHMARK
Cold Execution : 7.03 sec
Warm Execution : 30.77 sec
Speed-up        : 0.23x
COMMIT 2 COMPLETED

Spark UI
http://9775fb37b770:4040
